In [3]:
from Bio import SeqIO

for seq_record in SeqIO.parse("/Users/ailinzhang/Desktop/sequence_optimization_with_CodonBERT/data/SRR4163265.fasta", "fasta"):
# for seq_record in SeqIO.parse("../data/oligos.fasta", "fasta"):
    print(seq_record.id)
    print(repr(seq_record.seq))
    print(len(seq_record))
    break

SRR4163265.1
Seq('GGAGGTACCATAATGGTCGGTAATCCTAACTCTGAAAGTGATAAACACCTTGTT...GAT')
126


In [4]:
# Load model directly
from transformers import AutoTokenizer, AutoModelForPreTraining

tokenizer = AutoTokenizer.from_pretrained("lhallee/CodonBERT")
model = AutoModelForPreTraining.from_pretrained("lhallee/CodonBERT")

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt:   0%|          | 0.00/356 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/647 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/348M [00:00<?, ?B/s]

In [5]:
from transformers import AutoTokenizer, AutoModelForPreTraining, AutoModel
import torch
from Bio import SeqIO

# Load CodonBERT
tokenizer = AutoTokenizer.from_pretrained("lhallee/CodonBERT")
model = AutoModelForPreTraining.from_pretrained("lhallee/CodonBERT")

# Get one sequence from fasta
for seq_record in SeqIO.parse("../data/SRR4163265.fasta", "fasta"):
    dna_seq = str(seq_record.seq)
    break

print("Example sequence:", dna_seq[:60], "...")

# Tokenize
inputs = tokenizer(
    dna_seq,
    return_tensors="pt",
    max_length=512,   # CodonBERT has a context window, truncate if longer
    truncation=True,
    padding="max_length"
)

# Run inference
with torch.no_grad():
    outputs = model(**inputs)

# Masked LM predictions
mlm_logits = outputs.prediction_logits   # [batch_size, seq_len, vocab_size]
print("MLM logits shape:", mlm_logits.shape)

base_model = AutoModel.from_pretrained("lhallee/CodonBERT")

with torch.no_grad():
    emb_outputs = base_model(**inputs)

embeddings = emb_outputs.last_hidden_state  # [batch_size, seq_len, hidden_dim]
print("Embeddings shape:", embeddings.shape)



Example sequence: GGAGGTACCATAATGGTCGGTAATCCTAACTCTGAAAGTGATAAACACCTTGTTACACCG ...
MLM logits shape: torch.Size([1, 512, 69])
Embeddings shape: torch.Size([1, 512, 768])


In [6]:
embeddings

tensor([[[ 0.2680, -0.1236,  0.0787,  ..., -0.1736, -0.9255,  0.1316],
         [ 0.2702, -0.2250,  0.3532,  ...,  0.1391, -1.0764,  0.5688],
         [ 0.5499, -0.4428,  0.3693,  ...,  0.2403, -0.6261,  0.5298],
         ...,
         [ 0.8073, -0.5289,  0.4486,  ...,  0.1888, -0.7543,  0.6473],
         [ 0.9092, -0.5641,  0.2510,  ...,  0.2096, -0.6932,  0.6053],
         [ 0.8399, -0.6432,  0.3131,  ...,  0.3048, -0.6281,  0.6040]]])